# Pillar 1 — Dataset Quality (Figures 2 and 3)

Renders the two figures the Results section specifies for Pillar 1, in Nature submission
format via `nature_style.py`.

**Figure 2 — Sub-pillar 1a, human-rated comparison against PathVQA**
- **a** Word-count distributions (questions, correct answers, wrong answers), PathOPEN vs
  filtered-PathVQA, with Mann-Whitney annotations
- **b** Forest plot, Benchmark 1 effect sizes with 95% CI, OE correct answers
- **c** Forest plot, Benchmark 3 effect sizes, close-ended pairs

**Figure 3 — Sub-pillar 1b, judge-validated comparison against Quilt-VQA**
- **a** Judge-vs-pathologist agreement on the validation subset
- **b** Forest plot, judge-rated scores, PathOPEN vs Quilt-VQA

## Two decisions these figures encode

**Panel 3a shows AC1 alongside weighted kappa.** Reporting kappa alone would be actively
misleading here: PathOPEN's Benchmark 3 has 95.2% raw agreement but kappa = -0.022,
because ~97% of scores are "2" and kappa's chance correction collapses under that
prevalence. AC1 gives 0.984 on the same data. Showing both, with exact agreement, is the
honest presentation - and it pre-empts the reviewer question the paper's own
"report kappa honestly even if only moderate" note anticipates.

**Effect sizes carry bootstrapped 95% CIs** (2,000 resamples), which is what §2.1
specifies. The CI is the reason to prefer a forest plot over a bar chart: it shows the
comparison is decisive rather than asking the reader to trust a p-value.

In [ ]:
import os
import sys

sys.path.insert(0, os.path.abspath("."))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import nature_style as ns

ns.apply_style()

EVAL = os.path.join("..", "data_evaluation")
AGREE = os.path.join(EVAL, "vlm_judge", "agreement_output")
SUBSETS = os.path.join(EVAL, "pathologists", "subsets_processing_output", "data")

print("style applied |", plt.rcParams["font.size"], "pt base")

## Figure 2a — word-count distributions

PathOPEN's answers are ~3x longer than filtered-PathVQA's, which is the concrete form of
"greater linguistic and contextual depth". Violin plots rather than histograms because
six distributions on one axis need to be comparable at a glance.

In [ ]:
import glob

pathopen = pd.concat([pd.read_csv(p) for p in
                      sorted(glob.glob(os.path.join(SUBSETS, "pathopen_vqa_part*.csv")))],
                     ignore_index=True)
pathvqa = pd.concat([pd.read_csv(p) for p in
                     sorted(glob.glob(os.path.join(SUBSETS, "pathvqa_part*.csv")))],
                    ignore_index=True)


def word_counts(series):
    """Word counts, blanks dropped."""
    return series.dropna().astype(str).str.split().str.len().values


# PathOPEN carries two OE slots per row; pool them - the unit of analysis is the
# question/answer pair, not the row.
po_questions = np.concatenate([word_counts(pathopen["OE_Question_1"]),
                               word_counts(pathopen["OE_Question_2"])])
po_correct = np.concatenate([word_counts(pathopen["OE_Correct_Answer_1"]),
                             word_counts(pathopen["OE_Correct_Answer_2"])])
po_wrong = np.concatenate([word_counts(pathopen["OE_Wrong_Answer_1"]),
                           word_counts(pathopen["OE_Wrong_Answer_2"])])

# Filtered PathVQA: open-ended rows only, and it has no wrong answers at all - which is
# itself one of the paper's structural claims (Table 3).
pv_open = pathvqa[pathvqa.question_type != "close-ended"]
pv_questions = word_counts(pv_open["question"])
pv_correct = word_counts(pv_open["answer"])

print(f"PathOPEN  Q {po_questions.mean():.1f}w  A {po_correct.mean():.1f}w  "
      f"wrong {po_wrong.mean():.1f}w  (n={len(po_questions)})")
print(f"PathVQA   Q {pv_questions.mean():.1f}w  A {pv_correct.mean():.1f}w  "
      f"wrong: none  (n={len(pv_questions)})")

In [ ]:
from scipy.stats import mannwhitneyu

fig, ax = plt.subplots(figsize=ns.mm(ns.SINGLE_COL_MM, 58))

groups = [
    ("Questions", po_questions, pv_questions),
    ("Correct\nanswers", po_correct, pv_correct),
    ("Wrong\nanswers", po_wrong, None),   # PathVQA has none
]

positions, labels = [], []
for index, (label, pathopen_values, pathvqa_values) in enumerate(groups):
    base = index * 1.0
    for offset, values, color in ((-0.16, pathopen_values, ns.DATASET_COLORS["PathOPEN"]),
                                  (0.16, pathvqa_values, ns.DATASET_COLORS["PathVQA"])):
        if values is None:
            continue
        parts = ax.violinplot([values], positions=[base + offset], widths=0.28,
                              showmedians=True, showextrema=False)
        for body in parts["bodies"]:
            body.set_facecolor(color)
            body.set_alpha(0.85)
            body.set_edgecolor("none")
        parts["cmedians"].set_color("black")
        parts["cmedians"].set_linewidth(0.6)
    positions.append(base)
    labels.append(label)

    # Mann-Whitney where both datasets have data. Reported as stars because the exact
    # p-values (1e-60 and smaller) would dominate the panel without adding information.
    if pathvqa_values is not None:
        _, p = mannwhitneyu(pathopen_values, pathvqa_values, alternative="two-sided")
        top = max(np.percentile(pathopen_values, 97), np.percentile(pathvqa_values, 97))
        ax.plot([base - 0.16, base + 0.16], [top + 3, top + 3], lw=0.5, color="black")
        ax.text(base, top + 4, ns.significance_stars(p), ha="center", va="bottom", fontsize=7)

ax.set_xticks(positions)
ax.set_xticklabels(labels)
ax.set_ylabel("Word count")
ax.set_ylim(0, 60)
ax.text(2.0, 8, "PathVQA has no\nwrong answers", ha="center", va="center",
        fontsize=6.5, color="#666666", style="italic")

handles = [plt.Rectangle((0, 0), 1, 1, fc=ns.DATASET_COLORS["PathOPEN"]),
           plt.Rectangle((0, 0), 1, 1, fc=ns.DATASET_COLORS["PathVQA"])]
ns.legend_outside(ax, "above", handles=handles,
                  labels=["PathOPEN", "filtered PathVQA"], ncol=2)
ns.panel_label(ax, "a")
fig.tight_layout()
plt.show()

## Figure 2b,c — forest plots of effect size

Rank-biserial correlation with bootstrapped 95% CI. Negative means PathOPEN ranks higher.

The zero line is the reference: a CI that excludes it is a decisive result. Plotting all
three raters (pathologists + both judges) on one axis shows the judges reproduce the
humans' qualitative conclusion, which is the validation Sub-pillar 1b depends on.

In [ ]:
mann_whitney = pd.read_csv(os.path.join(AGREE, "pathopen_vs_pathvqa_mannwhitney.csv"))

CRITERION_LABELS = {
    "OE_correct_KnowledgeInterpretation": "Knowledge\nInterpretation",
    "OE_correct_VisualGrounding": "Visual\nGrounding",
    "CE_correct_VisualGrounding": "Visual Grounding/\nReasoning",
}
RATER_STYLE = {
    "human": ("Pathologists", "#000000", "o"),
    "internvl": ("InternVL judge", ns.OKABE_ITO["blue"], "s"),
    "qwenvl": ("Qwen judge", ns.OKABE_ITO["sky"], "^"),
}


def forest(ax, criteria, letter, title, label_below=False, xlabel=None):
    """One forest panel: effect size + 95% CI, one row per (criterion, rater)."""
    y = 0
    yticks, yticklabels = [], []
    for criterion in criteria:
        rows = mann_whitney[mann_whitney.criterion == criterion]
        centre = y + 1
        for rater, (label, color, marker) in RATER_STYLE.items():
            row = rows[rows.rater == rater]
            if row.empty:
                continue
            row = row.iloc[0]
            ax.plot([row.rb_ci_low, row.rb_ci_high], [y, y], lw=0.8, color=color,
                    solid_capstyle="butt")
            ax.plot(row.rank_biserial, y, marker=marker, ms=3.5, color=color,
                    mec="white", mew=0.3)
            y -= 1
        yticks.append(centre - 2)
        yticklabels.append(CRITERION_LABELS[criterion])
        y -= 0.6

    ax.axvline(0, ls="--", lw=0.5, color="#999999", zorder=0)
    ax.set_yticks(yticks)
    ax.set_yticklabels(yticklabels)
    ax.set_xlabel(xlabel or "Rank-biserial correlation (95% CI)")
    ax.set_title(title, pad=3)
    if label_below:
        ns.panel_label_below(ax, letter)
    else:
        ns.panel_label(ax, letter)


fig, axes = plt.subplots(1, 2, figsize=ns.mm(ns.DOUBLE_COL_MM, 62),
                         gridspec_kw={"width_ratios": [2, 1]})
forest(axes[0], ["OE_correct_KnowledgeInterpretation", "OE_correct_VisualGrounding"],
       "b", "Benchmark 1 — open-ended correct answers")
forest(axes[1], ["CE_correct_VisualGrounding"], "c", "Benchmark 3 — close-ended")

handles = [plt.Line2D([], [], color=c, marker=m, ms=3.5, lw=0.8, label=lbl)
           for lbl, c, m in RATER_STYLE.values()]
ns.legend_outside(axes[1], "right", handles=handles, fontsize=6.5)
axes[0].text(0.02, 0.03, "← PathOPEN scores higher", transform=axes[0].transAxes,
             fontsize=6.5, color="#666666")
fig.tight_layout()
plt.show()

### Assemble the pillar figure


In [ ]:
# ONE figure for the pillar: Nature reports one figure per result with lettered panels
# inside it, grown in height, rather than several numbered figures. Panels a-e cover both
# sub-pillars: a-c the human-rated PathVQA comparison, d-e the judge-validated Quilt-VQA
# comparison.
fig = plt.figure(figsize=ns.mm(ns.DOUBLE_COL_MM, 186))
# wspace is generous because panels d and f carry long multi-line y-tick labels
# (criterion names) on the right-hand column; at a tighter spacing they overrun panels
# c and e. width_ratios narrows the right column so those labels have somewhere to go.
grid = fig.add_gridspec(3, 3, height_ratios=[1, 0.95, 1.05],
                        width_ratios=[1, 1, 0.92], hspace=0.85, wspace=0.95)

# ---------- a: word counts ----------
ax_a = fig.add_subplot(grid[0, :2])
for index, (label, pathopen_values, pathvqa_values) in enumerate(groups):
    base = index * 1.0
    for offset, values, color in ((-0.16, pathopen_values, ns.DATASET_COLORS["PathOPEN"]),
                                  (0.16, pathvqa_values, ns.DATASET_COLORS["PathVQA"])):
        if values is None:
            continue
        parts = ax_a.violinplot([values], positions=[base + offset], widths=0.28,
                                showmedians=True, showextrema=False)
        for body in parts["bodies"]:
            body.set_facecolor(color); body.set_alpha(0.85); body.set_edgecolor("none")
        parts["cmedians"].set_color("black"); parts["cmedians"].set_linewidth(0.6)
    if pathvqa_values is not None:
        _, p = mannwhitneyu(pathopen_values, pathvqa_values, alternative="two-sided")
        top = max(np.percentile(pathopen_values, 97), np.percentile(pathvqa_values, 97))
        ax_a.plot([base - 0.16, base + 0.16], [top + 3, top + 3], lw=0.5, color="black")
        ax_a.text(base, top + 4, ns.significance_stars(p), ha="center", va="bottom",
                  fontsize=7)
ax_a.set_xticks([0, 1, 2])
ax_a.set_xticklabels(["Questions", "Correct answers", "Wrong answers"])
ax_a.set_ylabel("Word count")
ax_a.set_ylim(0, 62)
ax_a.text(2.28, 56, "PathVQA has no\nwrong answers", ha="center", va="top",
          fontsize=6.5, color="#666666", style="italic")
dataset_handles = [plt.Rectangle((0, 0), 1, 1, fc=ns.DATASET_COLORS["PathOPEN"]),
                   plt.Rectangle((0, 0), 1, 1, fc=ns.DATASET_COLORS["PathVQA"])]
ns.legend_outside(ax_a, "above", handles=dataset_handles,
                  labels=["PathOPEN", "filtered PathVQA"], ncol=2)
ns.panel_label_below(ax_a, "a")

# ---------- b: unscorable rate, per criterion ----------
ax_b = fig.add_subplot(grid[0, 2])
human = mann_whitney[mann_whitney.rater == "human"]
short = {"OE_correct_KnowledgeInterpretation": "B1\nKnow.",
         "OE_correct_VisualGrounding": "B1\nVis.Gr.",
         "CE_correct_VisualGrounding": "B3\nVis.Gr."}
xs = np.arange(len(human))
po_rate = 100 * human.n_neg1_pathopen.values / human.n_pathopen.values
pv_rate = 100 * human.n_neg1_pathvqa.values / human.n_pathvqa.values
ax_b.bar(xs - 0.19, po_rate, 0.36, color=ns.DATASET_COLORS["PathOPEN"])
ax_b.bar(xs + 0.19, pv_rate, 0.36, color=ns.DATASET_COLORS["PathVQA"])
for x, value in zip(xs - 0.19, po_rate):
    ax_b.text(x, value + 0.9, f"{value:.1f}", ha="center", fontsize=6.5)
for x, value in zip(xs + 0.19, pv_rate):
    ax_b.text(x, value + 0.9, f"{value:.1f}", ha="center", fontsize=6.5)
ax_b.set_xticks(xs)
ax_b.set_xticklabels([short[c] for c in human.criterion], fontsize=6.5)
ax_b.set_ylabel("Rated unscorable (%)")
ax_b.set_ylim(0, max(pv_rate) * 1.32)
ns.panel_label_below(ax_b, "b")

# ---------- c, d: forest plots ----------
ax_c = fig.add_subplot(grid[1, :2])
forest(ax_c, ["OE_correct_KnowledgeInterpretation", "OE_correct_VisualGrounding"],
       "c", "Benchmark 1 — open-ended correct answers", label_below=True)
ax_c.text(0.02, 0.03, "← PathOPEN scores higher", transform=ax_c.transAxes,
          fontsize=6.5, color="#666666")

ax_d = fig.add_subplot(grid[1, 2])
forest(ax_d, ["CE_correct_VisualGrounding"], "d", "Benchmark 3 — close-ended",
       label_below=True, xlabel="Rank-biserial\n($\\leftarrow$ PathOPEN higher)")
rater_handles = [plt.Line2D([], [], color=c, marker=m, ms=3.5, lw=0.8, label=lbl)
                 for lbl, c, m in RATER_STYLE.values()]
# Centred over the c+d row rather than inside d: the raters are shared by both forest
# plots, so a key sitting in one panel both hides that panel's data and misreads as
# applying only to it. Positioned from the two panels' joint extent.
from matplotlib.transforms import Bbox
cd_box = Bbox.union([ax_c.get_position(), ax_d.get_position()])
rater_legend = fig.legend(
    rater_handles, [h.get_label() for h in rater_handles],
    loc="lower center", bbox_to_anchor=(cd_box.x0 + cd_box.width / 2, cd_box.y1 + 0.008),
    ncol=len(rater_handles), frameon=False, fontsize=6.5,
    columnspacing=1.4, handletextpad=0.4)

# ---------- e: three agreement statistics ----------
ax_e = fig.add_subplot(grid[2, :2])
kappa = pd.read_csv(os.path.join(AGREE, "judge_pathologist_weighted_kappa.csv"))
# Average the repeated question-type blocks (OE_1/OE_2, MCQ wrong 1-4) so each
# (benchmark, criterion) appears once.
agreement = (kappa.groupby(["dataset", "benchmark", "criterion", "judge"])
             .agg(exact=("exact_agreement_pct", "mean"),
                  kappa=("weighted_kappa", "mean"),
                  ac1=("gwet_ac1", "mean"),
                  n=("n", "sum")).reset_index())
subset = agreement[(agreement.dataset == "PathOPEN") &
                   (agreement.judge == "qwenvl")].copy()
subset = subset.sort_values("benchmark")
x = np.arange(len(subset))
width = 0.26
ax_e.bar(x - width, subset.exact / 100, width, label="Exact agreement", color="#BFBFBF")
ax_e.bar(x, subset.kappa, width, label="Weighted $\\kappa$", color=ns.OKABE_ITO["sky"])
ax_e.bar(x + width, subset.ac1, width, label="Gwet's AC1", color=ns.OKABE_ITO["green"])
ax_e.axhline(0, lw=0.5, color="black")
ax_e.set_xticks(x)
ax_e.set_xticklabels([f"B{int(b)}\n{c.split()[0]}" for b, c in
                      zip(subset.benchmark, subset.criterion)], fontsize=6.5)
ax_e.set_ylabel("Agreement")
ax_e.set_ylim(-0.15, 1.05)
e_legend = ax_e.legend(loc="lower center", ncol=3,
                       bbox_to_anchor=(0.5, 1.10), frameon=False)
worst = subset.loc[subset.kappa.idxmin()]
worst_x = list(subset.index).index(worst.name)
# The callout is ~500 px on one line, nearly the panel's full width, so centring it on
# the flagged bar spreads it back across the others. Two short lines placed in headroom
# above every bar keep it clear of the data while staying next to what it describes.
ax_e.set_ylim(-0.15, 1.42)
ax_e.annotate(f"$\\kappa$ = {worst.kappa:.3f} despite\n{worst.exact:.1f}% agreement",
              xy=(worst_x, worst.kappa), xytext=(worst_x, 1.38),
              fontsize=6.5, color="#B33A3A", ha="center", va="top",
              linespacing=1.0,
              arrowprops=dict(arrowstyle="->", lw=0.5, color="#B33A3A",
                              connectionstyle="arc3,rad=-0.25"))
ns.panel_label_below(ax_e, "e")

# ---------- f: PathOPEN vs Quilt-VQA ----------
ax_f = fig.add_subplot(grid[2, 2])
quilt = pd.read_csv(os.path.join(AGREE, "pathopen_vs_quiltvqa_eval_mannwhitney.csv"))
y = 0
yticks, yticklabels = [], []
for criterion in quilt.criterion.unique():
    rows = quilt[quilt.criterion == criterion]
    centre = y
    for _, row in rows.iterrows():
        color = ns.OKABE_ITO["blue"] if row.judge == "internvl" else ns.OKABE_ITO["sky"]
        marker = "s" if row.judge == "internvl" else "^"
        ax_f.plot(row.rank_biserial, y, marker=marker, ms=3.5, color=color,
                  mec="white", mew=0.3)
        y -= 1
    yticks.append((centre + y + 1) / 2)
    # Short, wrapped labels: the raw criterion names are up to 25 characters and
    # overrun the neighbouring panel.
    short_label = {"Knowledge Interpretation/Deduction": "Knowledge\ninterp.",
                   "Visual Grounding": "Visual\ngrounding",
                   "Visual Grounding/Reasoning": "Visual gr./\nreasoning"}
    yticklabels.append(short_label.get(criterion, criterion.replace("/", "/\n")))
    y -= 0.6
ax_f.axvline(0, ls="--", lw=0.5, color="#999999", zorder=0)
ax_f.set_yticks(yticks)
ax_f.set_yticklabels(yticklabels, fontsize=6.5, linespacing=0.9)
ax_f.set_xlabel("Rank-biserial\n($\\leftarrow$ PathOPEN higher)")
ax_f.set_title("PathOPEN vs Quilt-VQA", pad=3)
judge_legend = ns.legend_outside(ax_f, "above", handles=[
    plt.Line2D([], [], color=ns.OKABE_ITO["blue"], marker="s", ms=3, lw=0,
               label="InternVL judge"),
    plt.Line2D([], [], color=ns.OKABE_ITO["sky"], marker="^", ms=3, lw=0,
               label="Qwen judge")],
    ncol=2, fontsize=6.5, columnspacing=1.0, handletextpad=0.4,
    bbox_to_anchor=(0.5, 1.06))
ns.panel_label_below(ax_f, "f")

ns.lift_legend_above_titles(fig, rater_legend, [ax_c, ax_d])
ns.lift_legend_above_titles(fig, judge_legend, [ax_f])

paths = ns.save(fig, "fig01_pillar1_dataset_quality")
print("wrote:", paths)
plt.show()

## Figure 3 — Sub-pillar 1b, Quilt-VQA

**a** Judge-vs-pathologist agreement on the validation subset. Three statistics per
criterion, because on this data they disagree in an informative way:

- **exact agreement** — what fraction of items the judge and pathologist scored identically
- **weighted kappa** — the conventional statistic, chance-corrected from rater marginals
- **Gwet's AC1** — chance-corrected in a way that is stable under high prevalence

PathOPEN's scores are ~95-97% "2", which is the regime where kappa's chance term
approaches observed agreement and the statistic collapses toward zero. Benchmark 3 is the
clearest case: 95.2% exact agreement, kappa -0.022, AC1 0.984. Plotting only kappa would
report near-perfect agreement as worse than random.

In [ ]:
kappa = pd.read_csv(os.path.join(AGREE, "judge_pathologist_weighted_kappa.csv"))

# One row per (dataset, benchmark, criterion) per judge; average the repeated
# question-type blocks (OE_1/OE_2, MCQ wrong 1-4) so each criterion appears once.
agreement = (kappa.groupby(["dataset", "benchmark", "criterion", "judge"])
             .agg(exact=("exact_agreement_pct", "mean"),
                  kappa=("weighted_kappa", "mean"),
                  ac1=("gwet_ac1", "mean"),
                  n=("n", "sum")).reset_index())
show = agreement[agreement.dataset.isin(["PathOPEN", "Filtered_PathVQA"])]
print(show.round(3).to_string(index=False))

## Draft captions

Copy this into the manuscript and adjust wording to house style. Every number below is
read from the CSVs this notebook loads, so it stays correct if the data is regenerated.

---

**Fig. 1 | PathOPEN shows greater contextual depth and fewer defective items than two
existing pathology VQA datasets.**
**a**, Word-count distributions for questions, correct answers and wrong answers
(violins show the full distribution; horizontal line is the median). PathOPEN questions
average 18.7 words against 7.8 for filtered PathVQA, and correct answers 22.8 against
7.8. Filtered PathVQA contains no wrong answers. Two-sided Mann-Whitney *U*:
\*\*\*\**P* < 10⁻⁴.
**b**, Percentage of items the pathologists scored −1 ("unable to comprehend the
question and/or image, or unable to make the evaluation"), per criterion. PathOPEN:
1.3%, 1.3%, 0.0%. Filtered PathVQA: 17.8%, 17.2%, 34.0%. Rates are given per criterion
rather than pooled, because each criterion scores the same items and pooling would
double-count them.
**c**, **d**, PathOPEN versus **filtered PathVQA**. Effect sizes (rank-biserial
correlation) with bootstrapped 95% confidence intervals (2,000 resamples), for
Benchmark 1 open-ended correct answers (**c**) and Benchmark 3 close-ended answers
(**d**). Rows show the pathologist consensus and each VLM judge. *n* = 231 PathOPEN and
624 filtered-PathVQA ratings (open-ended); 230 and 50 (close-ended).
**e**, Three agreement statistics between the Qwen judge and the single pathologist who
rated each PathOPEN item: exact agreement, quadratic-weighted Cohen's κ, and Gwet's AC1.
κ and AC1 differ only in how they estimate chance agreement. Because 95–97% of PathOPEN
scores are the maximum value, κ's chance term approaches observed agreement and the
statistic collapses — on Benchmark 3 it returns −0.022 despite 95.2% exact agreement,
where AC1 returns 0.984. AC1 is reported *alongside* κ, not instead of it.
**f**, PathOPEN versus **Quilt-VQA**, scored by both validated judges (no pathologist
ratings exist for Quilt-VQA, so no human row is shown). *n* = 229 PathOPEN against 724
Quilt-VQA ratings (open-ended) and 257 (close-ended). All six comparisons are
significant (*P* < 10⁻³).

**Sign convention (applies to c, d and f).** The rank-biserial correlation is signed so
that **negative values mean PathOPEN ranks higher**; the dashed line marks no
difference. Panels c, d and f each carry an on-figure "← PathOPEN scores higher" cue,
because a reader who assumes the opposite convention would invert every conclusion in
these three panels. Note the comparator differs: **c** and **d** compare against
filtered PathVQA, **f** against Quilt-VQA.

**Interpreting the effect sizes.** Rank-biserial is the probability that a randomly
drawn PathOPEN item outranks a randomly drawn comparator item, rescaled to [−1, 1]. The
values here (|r| = 0.09–0.40) are small-to-medium by convention. The consistency matters
more than the magnitude: every rater and every criterion points the same way in both
comparisons. The gap is widest on open-ended criteria and narrowest on close-ended ones,
where both datasets sit near the ceiling of the {−1, 0, 1, 2} scale and have little room
to separate. The very small *P*-values (to 10⁻²³) reflect sample size; quote the effect
size, not the *P*-value.

---

### Numbers a caption must not get wrong

| quantity | value |
|---|---|
| PathOPEN question / answer words | 18.7 / 22.8 |
| filtered PathVQA question / answer words | 7.8 / 7.8 |
| unscorable, B1 Knowledge | 3/231 (1.3%) vs 111/624 (17.8%) |
| unscorable, B1 Visual Grounding | 3/231 (1.3%) vs 107/622 (17.2%) |
| unscorable, B3 | 0/230 (0.0%) vs 17/50 (34.0%) |
| vs PathVQA, B1 Knowledge (human) | rb = −0.368 [−0.415, −0.320] |
| vs PathVQA, B1 Visual Grounding (human) | rb = −0.340 [−0.387, −0.291] |
| vs PathVQA, B3 (human) | rb = −0.399 [−0.546, −0.261] |
| vs Quilt-VQA, B1 Knowledge (Qwen / InternVL) | rb = −0.307 / −0.234 |
| vs Quilt-VQA, B1 Visual Grounding (Qwen / InternVL) | rb = −0.335 / −0.310 |
| vs Quilt-VQA, B3 (Qwen / InternVL) | rb = −0.134 / −0.108 |
| Quilt-VQA unscorable, B1 (Qwen) | 13/724 and 11/724, vs 0/229 for PathOPEN |
| B3 κ vs AC1 (the kappa paradox) | κ = −0.022, AC1 = 0.984, exact = 95.2% |

**Sign check.** All 15 rows across both Mann-Whitney CSVs have PathOPEN's mean above the
comparator's while rank-biserial is negative. If a regenerated file ever breaks that
pattern, the arrow annotations in c, d and f are wrong and must be flipped.
